<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/Aula_Passo_a_Passo_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q spacy scikit-learn gradio
!python -m spacy download pt_core_news_sm -q

import re
import unicodedata
import pandas as pd
import spacy
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado com sucesso.")

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado com sucesso.


In [2]:
imoveis = pd.DataFrame([
    ["AP01", "apartamento", "Aluguel", "Matão",        "Centro",                     2,   1300, False],
    ["AP02", "apartamento", "Aluguel", "Matão",        "Jardim Paraíso",             3,   1800, False],
    ["AP03", "apartamento", "Venda",   "Matão",        "Centro",                     2, 280000, True],
    ["CA01", "casa",        "Aluguel", "Matão",        "Vila Santa Cruz",            2,   1100, False],
    ["CA02", "casa",        "Venda",   "Matão",        "Residencial Olívio Benassi", 3, 420000, True],
    ["CA03", "casa",        "Venda",   "Matão",        "Jardim Bela Vista",          2, 250000, True],
    ["KT01", "kitnet",      "Aluguel", "Matão",        "Centro",                     1,    750, False],
    ["AP04", "apartamento", "Aluguel", "Araraquara",   "Centro",                     2,   1600, False],
    ["AP05", "apartamento", "Venda",   "Araraquara",   "Fonte",                      3, 390000, True],
    ["CA04", "casa",        "Aluguel", "Araraquara",   "Vila Xavier",                3,   2200, False],
    ["CA05", "casa",        "Venda",   "Araraquara",   "Jardim Primavera",           2, 310000, True],
    ["CA06", "casa",        "Venda",   "Taquaritinga", "Centro",                     3, 350000, True],
    ["CA07", "casa",        "Aluguel", "Taquaritinga", "Jardim Buscardi",            2,   1000, False],
    ["AP06", "apartamento", "Venda",   "Taquaritinga", "Centro",                     2, 230000, True],
    ["TE01", "terreno",     "Venda",   "Matão",        "Jardim Paraíso",             0, 120000, True],
], columns=["codigo", "tipo", "finalidade", "cidade", "bairro", "quartos", "valor", "aceita_financiamento"])

print(imoveis.shape)
imoveis.head()

(15, 8)


,codigo,tipo,finalidade,cidade,bairro,quartos,valor,aceita_financiamento
0,AP01,apartamento,Aluguel,Matão,Centro,2,1300,False
1,AP02,apartamento,Aluguel,Matão,Jardim Paraíso,3,1800,False
2,AP03,apartamento,Venda,Matão,Centro,2,280000,True
3,CA01,casa,Aluguel,Matão,Vila Santa Cruz,2,1100,False
4,CA02,casa,Venda,Matão,Residencial Olívio Benassi,3,420000,True


In [4]:
filtro = (imoveis["finalidade"] == "Aluguel") & (imoveis["cidade"] == "Matão")
imoveis[filtro]

,codigo,tipo,finalidade,cidade,bairro,quartos,valor,aceita_financiamento
0,AP01,apartamento,Aluguel,Matão,Centro,2,1300,False
1,AP02,apartamento,Aluguel,Matão,Jardim Paraíso,3,1800,False
3,CA01,casa,Aluguel,Matão,Vila Santa Cruz,2,1100,False
6,KT01,kitnet,Aluguel,Matão,Centro,1,750,False


In [5]:
def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

print(normalizar("Qual o HORÁRIO de atendimento?"))
print(normalizar("Olá, boa tarde!"))

qual o horario de atendimento?
ola, boa tarde!


In [6]:
REGRAS = {
    "saudacao":   (["oi", "ola", "bom dia", "boa tarde", "boa noite"],
                   "Olá! Sou o assistente virtual da imobiliária. Posso ajudar você a comprar ou alugar um imóvel."),
    "horario":    (["horario", "abre", "funciona", "expediente"],
                   "Atendemos de segunda a sexta, das 8h às 18h, e aos sábados, das 8h às 12h."),
    "fgts":       (["fgts"],
                   "Sim! O FGTS pode ser usado na compra do imóvel, desde que o comprador atenda às regras da Caixa."),
    "documentos": (["documento", "documentos", "documentacao"],
                   "Para alugar: RG, CPF, comprovante de renda e de residência. Para comprar, o corretor envia a lista completa."),
    "despedida":  (["obrigado", "obrigada", "tchau", "valeu"],
                   "Eu que agradeço! Quando quiser, é só chamar."),
}

def responder_regras(mensagem):
    texto = normalizar(mensagem)   # normalização (Seção 2)
    palavras = set(re.findall(r"\w+", texto))   # tokenização com expressão regular
    for nome, (gatilhos, resposta) in REGRAS.items():
        for gatilho in gatilhos:
            if (" " in gatilho and gatilho in texto) or gatilho in palavras:
                return resposta
    return None

for m in ["Oi, tudo bem?", "Qual o horário de vocês?", "Posso usar meu FGTS?", "Quero alugar um apartamento"]:
    print(f"{m:30} -> {responder_regras(m)}")

Oi, tudo bem?                  -> Olá! Sou o assistente virtual da imobiliária. Posso ajudar você a comprar ou alugar um imóvel.
Qual o horário de vocês?       -> Atendemos de segunda a sexta, das 8h às 18h, e aos sábados, das 8h às 12h.
Posso usar meu FGTS?           -> Sim! O FGTS pode ser usado na compra do imóvel, desde que o comprador atenda às regras da Caixa.
Quero alugar um apartamento    -> None


In [7]:
texto = normalizar("Tem apartamento de dois quartos?")
print("'oi' aparece como pedaço do texto?  ", "oi" in texto)
print("'oi' aparece como palavra inteira?  ", "oi" in set(re.findall(r"\w+", texto)))

'oi' aparece como pedaço do texto?   True
'oi' aparece como palavra inteira?   False


In [10]:
imoveis_ex = ["o apartamento", "a casa", "o sobrado", "o studio", "o terreno", "a cobertura"]

modelos_compra = ["Quero comprar {}, aceita financiamento?", "Qual o valor de venda d{}?",

                  "Tenho interesse em adquirir {} à vista.", "Posso usar o FGTS para comprar {}?",

                  "Qual a entrada mínima para financiar {}?"]

modelos_aluguel = ["{} ainda está disponível para alugar?", "Qual o valor do aluguel d{}?",

                   "Aceitam fiador para alugar {}?", "{} para locação aceita pet?",

                   "Qual o prazo do contrato de aluguel d{}?"]

def montar(modelo, imovel):

    if "d{}" in modelo:

        return modelo.replace("d{}", "d" + imovel)

    frase = modelo.format(imovel)

    return frase[0].upper() + frase[1:]

textos_intencao = [montar(m, i) for m in modelos_compra for i in imoveis_ex] + [montar(m, i) for m in modelos_aluguel for i in imoveis_ex]
rotulos_intencao = ["Compra"] * 30 + ["Aluguel"] * 30

PALAVRAS_NEGACAO = {"não", "nunca", "jamais", "nada", "ninguém"}

def lematizar(texto):

    doc = nlp(texto)

    return " ".join(

        token.lemma_.lower() for token in doc

        if not token.is_punct and (not token.is_stop or token.lower_ in PALAVRAS_NEGACAO)

    )

vetorizador = TfidfVectorizer()   # TF-IDF (aula de Features)

X_intencao = vetorizador.fit_transform([lematizar(t) for t in textos_intencao])   # lematização + vetorização

modelo_intencao = LogisticRegression(max_iter=1000)   # Regressão Logística (ML Parte 1)

modelo_intencao.fit(X_intencao, rotulos_intencao)   # treino (ML Parte 1)

print("Mensagens de treino:", len(textos_intencao))

print("Vocabulário:", len(vetorizador.get_feature_names_out()), "palavras")


Mensagens de treino: 60
Vocabulário: 26 palavras


In [11]:
LIMIAR = 0.6

def classificar_intencao(mensagem):

    vetor = vetorizador.transform([lematizar(mensagem)])

    probabilidades = modelo_intencao.predict_proba(vetor)[0]

    posicao = probabilidades.argmax()

    return modelo_intencao.classes_[posicao], probabilidades[posicao]

for m in ["Quero alugar um apartamento de 2 quartos em Matão",

          "Gostaria de adquirir um imóvel em Matão",

          "Procuro casa para locação em Taquaritinga",

          "Oi, bom dia!",

          "Quero morar em São Carlos"]:

    intencao, confianca = classificar_intencao(m)

    decisao = intencao if confianca >= LIMIAR else "(não sei)"

    print(f"{m:52} {intencao:8} {confianca:.2f}  -> {decisao}")


Quero alugar um apartamento de 2 quartos em Matão    Aluguel  0.70  -> Aluguel
Gostaria de adquirir um imóvel em Matão              Compra   0.69  -> Compra
Procuro casa para locação em Taquaritinga            Aluguel  0.65  -> Aluguel
Oi, bom dia!                                         Compra   0.52  -> (não sei)
Quero morar em São Carlos                            Compra   0.52  -> (não sei)


In [12]:
for m in ["Quero alugar um apartamento de 2 quartos em Matão",

          "Procuro casa para locação em Taquaritinga",

          "Oi, bom dia!"]:

    doc = nlp(m)

    print(f"{m:52} {[(ent.text, ent.label_) for ent in doc.ents]}")

Quero alugar um apartamento de 2 quartos em Matão    [('Quero', 'LOC'), ('Matão', 'LOC')]
Procuro casa para locação em Taquaritinga            [('Procuro', 'PER'), ('Taquaritinga', 'LOC')]
Oi, bom dia!                                         [('Oi', 'ORG')]


In [14]:
TIPOS = {"apartamento": "apartamento", "apto": "apartamento", "casa": "casa",
         "kitnet": "kitnet", "terreno": "terreno", "lote": "terreno"}
CIDADES = ["Matão", "Araraquara", "Taquaritinga"]

def extrair_entidades(mensagem):
    texto = normalizar(mensagem)   # normalização (Seção 2)
    palavras = set(re.findall(r"\w+", texto))
    entidades = {"tipo": None, "cidade": None, "quartos": None, "valor_max": None}

    for palavra, tipo in TIPOS.items():   # tipo: lista de palavras
        if palavra in palavras:
            entidades["tipo"] = tipo

    for cidade in CIDADES:   # cidade: lista (no lugar do NER)
        if normalizar(cidade) in texto:
            entidades["cidade"] = cidade

    achou = re.search(r"(\d+)\s*quartos?", texto)   # quartos: expressão regular
    if achou:
        entidades["quartos"] = int(achou.group(1))

    achou = re.search(r"ate\s*(?:r\$\s*)?(\d+(?:\.\d{3})*)\s*(mil)?", texto)   # valor: expressão regular
    if achou:
        valor = int(achou.group(1).replace(".", ""))
        if achou.group(2):
            valor = valor * 1000
        entidades["valor_max"] = valor

    return entidades

for m in ["Quero alugar um apartamento de 2 quartos em Matão",
          "Tem casa para comprar em Araraquara até 350 mil?",
          "Tem kitnet para alugar até R$ 800?",
          "Oi, bom dia!"]:
    print(f"{m:50} {extrair_entidades(m)}")

Quero alugar um apartamento de 2 quartos em Matão  {'tipo': 'apartamento', 'cidade': 'Matão', 'quartos': 2, 'valor_max': None}
Tem casa para comprar em Araraquara até 350 mil?   {'tipo': 'casa', 'cidade': 'Araraquara', 'quartos': None, 'valor_max': 350000}
Tem kitnet para alugar até R$ 800?                 {'tipo': 'kitnet', 'cidade': None, 'quartos': None, 'valor_max': 800}
Oi, bom dia!                                       {'tipo': None, 'cidade': None, 'quartos': None, 'valor_max': None}


In [15]:
def buscar_imoveis(finalidade, entidades):
    resultado = imoveis
    if finalidade:
        resultado = resultado[resultado["finalidade"] == finalidade]
    if entidades["tipo"]:
        resultado = resultado[resultado["tipo"] == entidades["tipo"]]
    if entidades["cidade"]:
        resultado = resultado[resultado["cidade"] == entidades["cidade"]]
    if entidades["quartos"]:
        resultado = resultado[resultado["quartos"] >= entidades["quartos"]]
    if entidades["valor_max"]:
        resultado = resultado[resultado["valor"] <= entidades["valor_max"]]
    return resultado

def formatar_valor(valor, finalidade):
    texto = f"R$ {valor:,.0f}".replace(",", ".")
    return texto + "/mês" if finalidade == "Aluguel" else texto

def montar_resposta(resultado):
    if len(resultado) == 0:
        return "Não encontrei imóveis com essas características. Quer tentar outra cidade ou outro valor?"
    linhas = [f"Encontrei {len(resultado)} imóvel(is):"]
    for _, imovel in resultado.head(3).iterrows():
        linhas.append(f"• {imovel['codigo']}: {imovel['tipo']}, {imovel['quartos']} quarto(s), "
                      f"{imovel['bairro']}, {imovel['cidade']}: {formatar_valor(imovel['valor'], imovel['finalidade'])}")
    if len(resultado) > 3:
        linhas.append(f"...e mais {len(resultado) - 3}. Quer que um corretor entre em contato?")
    return "\n".join(linhas)

ents = extrair_entidades("Quero alugar um apartamento em Matão")
print(montar_resposta(buscar_imoveis("Aluguel", ents)))

Encontrei 2 imóvel(is):
• AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
• AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês


In [16]:
def responder_pedido(finalidade, entidades):

    resultado = buscar_imoveis(finalidade, entidades)

    if len(resultado) == 0 and entidades["tipo"]:

        sem_tipo = entidades.copy()

        sem_tipo["tipo"] = None

        alternativas = buscar_imoveis(finalidade, sem_tipo)

        if len(alternativas) > 0:

            return (f"Não encontrei {entidades['tipo']} com essas características, "

                    "mas tenho outras opções.\n" + montar_resposta(alternativas))

    return montar_resposta(resultado)

ents = extrair_entidades("Quero alugar um apartamento em Taquaritinga")

print(responder_pedido("Aluguel", ents))


Não encontrei apartamento com essas características, mas tenho outras opções.
Encontrei 1 imóvel(is):
• CA07: casa, 2 quarto(s), Jardim Buscardi, Taquaritinga: R$ 1.000/mês


In [17]:
FINALIDADE = {"Compra": "Venda", "Aluguel": "Aluguel"}

def chatbot(mensagem):
    resposta_regra = responder_regras(mensagem)   # Camada 1: regras (Seção 2)
    entidades = extrair_entidades(mensagem)   # Camada 3: entidades (Seção 4)
    tem_detalhes = any(entidades.values())
    intencao, confianca = classificar_intencao(mensagem)   # Camada 2: intenção (Seção 3)

    if resposta_regra and not tem_detalhes:
        return resposta_regra
    if not tem_detalhes and confianca < LIMIAR:   # limiar de confiança (Seção 3.1)
        return "Desculpe, não entendi. Você procura um imóvel para comprar ou para alugar?"
    if confianca < LIMIAR:
        return "Entendi o que você procura! É para comprar ou para alugar?"

    resposta = responder_pedido(FINALIDADE[intencao], entidades)   # Camada 4: busca (Seção 5)
    if resposta_regra:
        resposta = resposta_regra + "\n" + resposta
    return resposta

conversa = ["Oi, boa tarde!",
            "Quero alugar um apartamento em Matão",
            "Tem casa para comprar em Araraquara até 350 mil?",
            "Procuro casa para locação em Taquaritinga",
            "Quero financiar uma casa com 3 quartos",
            "Tem casa em Matão?",
            "Posso usar meu FGTS?",
            "Quero morar em São Carlos",
            "Obrigado!"]
for m in conversa:
    print("Lead:", m)
    print("Bot: ", chatbot(m).replace("\n", "\n      "))
    print()

Lead: Oi, boa tarde!
Bot:  Olá! Sou o assistente virtual da imobiliária. Posso ajudar você a comprar ou alugar um imóvel.

Lead: Quero alugar um apartamento em Matão
Bot:  Encontrei 2 imóvel(is):
      • AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

Lead: Tem casa para comprar em Araraquara até 350 mil?
Bot:  Encontrei 1 imóvel(is):
      • CA05: casa, 2 quarto(s), Jardim Primavera, Araraquara: R$ 310.000

Lead: Procuro casa para locação em Taquaritinga
Bot:  Encontrei 1 imóvel(is):
      • CA07: casa, 2 quarto(s), Jardim Buscardi, Taquaritinga: R$ 1.000/mês

Lead: Quero financiar uma casa com 3 quartos
Bot:  Encontrei 2 imóvel(is):
      • CA02: casa, 3 quarto(s), Residencial Olívio Benassi, Matão: R$ 420.000
      • CA06: casa, 3 quarto(s), Centro, Taquaritinga: R$ 350.000

Lead: Tem casa em Matão?
Bot:  Entendi o que você procura! É para comprar ou para alugar?

Lead: Posso usar meu FGTS?
Bot:

In [18]:
for m in ["Quero alugar um apartamento em Matão", "E com 3 quartos?"]:
    print("Lead:", m)
    print("Bot: ", chatbot(m).replace("\n", "\n      "))
    print()

Lead: Quero alugar um apartamento em Matão
Bot:  Encontrei 2 imóvel(is):
      • AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

Lead: E com 3 quartos?
Bot:  Entendi o que você procura! É para comprar ou para alugar?



In [19]:
for m in ["Em Matão até 200 mil tem algo?", "Comprar"]:
    print("Lead:", m)
    print("Bot: ", chatbot(m).replace("\n", "\n      "))
    print()

Lead: Em Matão até 200 mil tem algo?
Bot:  Entendi o que você procura! É para comprar ou para alugar?

Lead: Comprar
Bot:  Encontrei 8 imóvel(is):
      • AP03: apartamento, 2 quarto(s), Centro, Matão: R$ 280.000
      • CA02: casa, 3 quarto(s), Residencial Olívio Benassi, Matão: R$ 420.000
      • CA03: casa, 2 quarto(s), Jardim Bela Vista, Matão: R$ 250.000
      ...e mais 5. Quer que um corretor entre em contato?



In [20]:
def juntar_pedido(mensagens):
    finalidade = None
    entidades = {"tipo": None, "cidade": None, "quartos": None, "valor_max": None}
    for m in mensagens:
        detalhes = extrair_entidades(m)   # Camada 3: entidades (Seção 4)
        if responder_regras(m) and not any(detalhes.values()):   # Camada 1: regras (Seção 2)
            continue
        intencao, confianca = classificar_intencao(m)   # Camada 2: intenção (Seção 3)
        if confianca >= LIMIAR:   # limiar de confiança (Seção 3.1)
            finalidade = FINALIDADE[intencao]
        for chave, valor in detalhes.items():
            if valor:
                entidades[chave] = valor
    return finalidade, entidades

def chatbot(mensagem, anteriores=None):
    if anteriores is None:
        anteriores = []
    resposta_regra = responder_regras(mensagem)
    tem_detalhes = any(extrair_entidades(mensagem).values())
    intencao, confianca = classificar_intencao(mensagem)

    if resposta_regra and not tem_detalhes:
        return resposta_regra
    if not tem_detalhes and confianca < LIMIAR:
        return "Desculpe, não entendi. Você procura um imóvel para comprar ou para alugar?"

    finalidade, entidades = juntar_pedido(anteriores + [mensagem])   # memória: todas as mensagens do lead
    if finalidade is None:
        return "Entendi o que você procura! É para comprar ou para alugar?"

    resposta = responder_pedido(finalidade, entidades)   # Camada 4: busca (Seção 5)
    if resposta_regra:
        resposta = resposta_regra + "\n" + resposta
    return resposta

In [21]:
anteriores = []
for m in ["Em Matão até 200 mil tem algo?", "Comprar",
          "Quero alugar um apartamento", "E com 3 quartos?",
          "E em Araraquara?"]:
    print("Lead:", m)
    print("Bot: ", chatbot(m, anteriores).replace("\n", "\n      "))
    print()
    anteriores.append(m)

Lead: Em Matão até 200 mil tem algo?
Bot:  Entendi o que você procura! É para comprar ou para alugar?

Lead: Comprar
Bot:  Encontrei 1 imóvel(is):
      • TE01: terreno, 0 quarto(s), Jardim Paraíso, Matão: R$ 120.000

Lead: Quero alugar um apartamento
Bot:  Encontrei 2 imóvel(is):
      • AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

Lead: E com 3 quartos?
Bot:  Encontrei 1 imóvel(is):
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

Lead: E em Araraquara?
Bot:  Não encontrei apartamento com essas características, mas tenho outras opções.
      Encontrei 1 imóvel(is):
      • CA04: casa, 3 quarto(s), Vila Xavier, Araraquara: R$ 2.200/mês



In [22]:
def mensagens_do_lead(historico):

    anteriores = []

    for item in historico:

        if isinstance(item, dict):                 # formato {"role": ..., "content": ...}

            if item["role"] != "user":

                continue

            conteudo = item["content"]

        else:                                      # formato [mensagem do lead, resposta do bot]

            conteudo = item[0]

        if isinstance(conteudo, list):             # texto dentro de uma lista de partes

            conteudo = " ".join(parte["text"] for parte in conteudo

                                if isinstance(parte, dict) and "text" in parte)

        if isinstance(conteudo, str) and conteudo:

            anteriores.append(conteudo)

    return anteriores

def responder(mensagem, historico):

    return chatbot(mensagem, mensagens_do_lead(historico))   # o bot (Seção 7.1), com a memória do histórico

tela = gr.ChatInterface(   # Gradio: monta a janela de chat

    fn=responder,

    title="Assistente Virtual — Imobiliária",

    description="Pergunte sobre imóveis para comprar ou alugar em Matão, Araraquara e Taquaritinga.",

    examples=["Oi, bom dia!", "Quero alugar um apartamento em Matão",

              "Tem casa para comprar em Araraquara até 350 mil?"],

)

tela.launch(share=True, debug=True)   # Gradio: coloca no ar com link público


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://442f2cc4ac504d2f57.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://442f2cc4ac504d2f57.gradio.live
